# Output 6 — Probability approach

Excel analogue: **Output 6 - Prob (if applicable)** / **Probability approach**.
Paths are baseline, A1 historical, and the Chart Data most-extreme shock.
Distress probabilities use Excel `NORMDIST` (CPIA, growth, reserves/imports,
remittances, world growth), not `Φ((ratio − T) / T)`.

See `docs/11-scenario.qmd`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.dsa import load_core
from lic_dsf.output import (
    external_debt_scenarios_table,
    probabilities_table,
    probability_panel,
)
from lic_dsf.pv import load_input7_residual_params
from lic_dsf.rating import load_ci_summary, most_extreme_shock_id
from lic_dsf.scenario import (
    ProbabilityAssumptions,
    load_distress_covariates,
)
from lic_dsf.stress import (
    load_input6_standard,
    run_a1_historical_external,
    run_standard_external_stress,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORKBOOK

PosixPath('/home/sravan/excel-grapher/py-lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

In [2]:
macro, external, ext_base, _pub_base = load_core(WORKBOOK)
ci = load_ci_summary(WORKBOOK)
input6 = load_input6_standard(WORKBOOK)
residual = load_input7_residual_params(WORKBOOK)
external_stress = run_standard_external_stress(macro, external, input6, residual)
historical = run_a1_historical_external(macro, external, residual)

first_proj = int(macro.inputs.first_projection_year)
rating_years = list(range(first_proj, first_proj + 11))
panel_years = [int(y) for y in ext_base.years if int(y) >= first_proj]
mx_sid = most_extreme_shock_id(
    {sid: book.pv_ppg_external_to_gdp() for sid, book in external_stress.items()},
    ci.thresholds.pv_debt_to_gdp,
    rating_years,
)
(
    ci.country,
    ci.thresholds.pv_debt_to_gdp,
    mx_sid,
    panel_years[0],
    panel_years[-1],
    len(panel_years),
)

('Ghana', 40.0, 'B3_Exports', 2024, 2044, 21)

## Output 6 panels

Excel Output 6 charts plot 11 years; the **Probability approach** tables
run the full projection (`H:AB`, 2024–2044 here). For each indicator we
print two blocks like the workbook sheet:

1. **External debt scenarios** — baseline, A1 historical, and MX shock levels,
   plus the CI threshold and borderline bands (`T × (1 ± bw/2)`).
2. **Probabilities** — Excel `NORMDIST` distress probabilities (percent) and
   the template probability cutoff (`O64:O67`).

The MX shock is Chart Data’s most-extreme selector (PV/GDP, years 2–11),
then the same shock book is used for all four indicators.

In [3]:
from IPython.display import Markdown, display

INDICATORS = (
    ("PV of debt-to-GDP ratio", "pv_ppg_external_to_gdp", "pv_debt_to_gdp"),
    ("PV of debt-to-exports ratio", "pv_ppg_external_to_exports", "pv_debt_to_exports"),
    (
        "Debt service-to-exports ratio",
        "ppg_debt_service_to_exports",
        "debt_service_to_exports",
    ),
    (
        "Debt service-to-revenue ratio",
        "ppg_debt_service_to_revenue",
        "debt_service_to_revenue",
    ),
)

assumptions = ProbabilityAssumptions(bandwidth=0.1)
covariates = load_distress_covariates(WORKBOOK)
thresh = ci.thresholds.as_dict()
mx_book = external_stress[mx_sid]

out_6: dict[str, pd.DataFrame] = {}
for title, method, indicator in INDICATORS:
    panel = probability_panel(
        {
            "baseline": getattr(ext_base, method)().reindex(panel_years),
            "historical": getattr(historical, method)().reindex(panel_years),
            "mx_shock": getattr(mx_book, method)().reindex(panel_years),
        },
        float(thresh[indicator]),
        indicator=indicator,
        assumptions=assumptions,
        covariates=covariates,
    )
    out_6[indicator] = panel
    display(Markdown(f"### {title}"))
    display(Markdown("**External debt scenarios**"))
    display(external_debt_scenarios_table(panel))
    display(Markdown("**Probabilities**"))
    display(probabilities_table(panel))

### PV of debt-to-GDP ratio

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044
baseline level,44.8846,43.1514,41.2265,40.1941,38.6043,37.6952,35.4235,32.9897,31.9558,31.4151,31.2078,31.1940,30.9871,30.6630,30.4069,30.2763,29.7314,29.2787,28.5711,28.0691,27.5761
baseline prob,0.1842,0.1771,0.1695,0.1655,0.1595,0.1561,0.1479,0.1394,0.1359,0.1341,0.1334,0.1334,0.1327,0.1316,0.1308,0.1303,0.1286,0.1271,0.1249,0.1233,0.1217
historical level,44.8846,46.6083,47.2008,48.1508,48.3849,49.1587,48.8022,46.4066,44.4077,42.1843,39.5685,37.5609,35.6586,33.9615,32.4772,31.2851,29.7144,28.2941,26.7555,25.3590,23.9386
historical prob,0.1842,0.1913,0.1938,0.1979,0.1989,0.2022,0.2007,0.1905,0.1822,0.1733,0.1632,0.1557,0.1488,0.1428,0.1377,0.1337,0.1285,0.1240,0.1192,0.1150,0.1108
mx_shock level,44.8846,48.1736,56.0671,54.9660,53.2750,52.2214,49.1015,44.5914,41.5529,39.0202,36.8239,35.2940,34.4780,33.5594,32.7488,32.1274,31.1580,30.3668,29.3858,28.6754,28.0205
mx_shock prob,0.1842,0.1980,0.2335,0.2284,0.2206,0.2158,0.2020,0.1830,0.1708,0.1611,0.1530,0.1475,0.1446,0.1414,0.1386,0.1365,0.1333,0.1306,0.1275,0.1252,0.1231
threshold,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000,40.0000
lower_band,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000
upper_band,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000,42.0000


### PV of debt-to-exports ratio

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044
baseline level,109.0647,103.4689,99.5562,98.4192,95.2973,93.7093,89.4654,84.9560,83.8739,83.5280,83.6537,83.9265,83.7209,83.2823,83.0684,83.3238,82.4575,81.6315,80.2796,79.4748,78.5439
baseline prob,0.1128,0.1090,0.1064,0.1057,0.1037,0.1026,0.0999,0.0971,0.0965,0.0962,0.0963,0.0965,0.0964,0.0961,0.0960,0.0961,0.0956,0.0951,0.0943,0.0938,0.0932
historical level,109.0647,111.7581,113.9833,117.9021,119.4413,122.2074,123.2547,119.5074,116.5560,112.1617,106.0650,101.0564,96.3425,92.2411,88.7242,86.1003,82.4103,78.8863,75.1782,71.8015,68.1832
historical prob,0.1128,0.1147,0.1163,0.1190,0.1201,0.1221,0.1229,0.1202,0.1181,0.1150,0.1108,0.1074,0.1043,0.1017,0.0995,0.0978,0.0956,0.0934,0.0912,0.0893,0.0872
mx_shock level,109.0647,128.8970,167.7947,166.7977,162.9846,160.8882,153.6871,142.3129,135.1625,128.5763,122.3295,117.6809,115.4446,112.9611,110.8760,109.5773,107.0935,104.9258,102.3279,100.6209,98.9083
mx_shock prob,0.1128,0.1271,0.1585,0.1576,0.1543,0.1526,0.1465,0.1374,0.1318,0.1268,0.1222,0.1189,0.1173,0.1155,0.1141,0.1132,0.1115,0.1100,0.1083,0.1071,0.1060
threshold,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000,180.0000
lower_band,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000,171.0000
upper_band,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000,189.0000


### Debt service-to-exports ratio

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044
baseline level,17.1681,16.5074,15.1705,16.2756,18.5608,15.8922,19.6644,19.4605,16.5719,15.0800,13.3194,14.4836,14.1429,14.1152,13.4201,13.6872,14.4874,14.4273,14.7620,13.9394,13.4887
baseline prob,0.1660,0.1602,0.1490,0.1582,0.1786,0.1550,0.1890,0.1870,0.1608,0.1482,0.1343,0.1434,0.1407,0.1405,0.1350,0.1371,0.1434,0.1429,0.1456,0.1391,0.1356
historical level,17.1681,16.4580,15.5928,16.9478,19.4128,17.1687,22.1874,23.2460,21.4122,20.6039,19.3134,19.8376,18.7628,17.8789,16.4184,15.6799,15.3671,14.6884,14.4940,13.4478,12.8162
historical prob,0.1660,0.1598,0.1525,0.1641,0.1866,0.1660,0.2141,0.2251,0.2061,0.1981,0.1856,0.1906,0.1804,0.1723,0.1595,0.1532,0.1506,0.1450,0.1435,0.1353,0.1305
mx_shock level,17.1681,18.5538,20.2982,23.7881,26.6881,23.2948,30.2001,33.9561,29.9512,27.6701,25.0110,24.4199,20.5295,20.2847,19.1235,19.1281,19.7657,19.3034,19.4055,18.0995,17.3208
mx_shock prob,0.1660,0.1785,0.1951,0.2309,0.2634,0.2257,0.3055,0.3536,0.3024,0.2748,0.2443,0.2378,0.1974,0.1950,0.1838,0.1839,0.1899,0.1855,0.1865,0.1743,0.1673
threshold,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000,15.0000
lower_band,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500,14.2500
upper_band,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500,15.7500


### Debt service-to-revenue ratio

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044
baseline level,39.0015,37.8686,33.7022,35.3500,40.2087,34.3537,42.3044,41.5353,35.0544,31.7428,27.9829,30.4390,29.7157,29.6099,28.0757,28.4743,29.9879,29.7722,30.2916,28.4854,27.4661
baseline prob,0.4143,0.3978,0.3391,0.3619,0.4320,0.3480,0.4630,0.4516,0.3578,0.3126,0.2646,0.2956,0.2863,0.2849,0.2658,0.2707,0.2898,0.2870,0.2937,0.2708,0.2583
historical level,39.0015,37.7552,34.6404,36.8101,42.0544,37.1131,47.7321,49.6149,45.2931,43.3705,40.5756,41.6911,39.4226,37.5049,34.3482,32.6199,31.8088,30.3109,29.7417,27.4807,26.0967
historical prob,0.4143,0.3962,0.3520,0.3826,0.4593,0.3870,0.5440,0.5718,0.5076,0.4789,0.4374,0.4539,0.4204,0.3926,0.3480,0.3244,0.3135,0.2939,0.2866,0.2585,0.2420
mx_shock level,39.0015,38.1429,36.3863,41.6902,46.6513,40.6322,52.4245,58.4796,51.1219,46.9977,42.3994,41.4114,34.8056,34.3352,32.2822,32.1094,33.0132,32.1426,32.1310,29.8447,28.4589
mx_shock prob,0.4143,0.4018,0.3766,0.4539,0.5279,0.4382,0.6126,0.6960,0.5938,0.5330,0.4644,0.4498,0.3543,0.3478,0.3198,0.3175,0.3297,0.3180,0.3178,0.2879,0.2705
threshold,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000,18.0000
lower_band,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000,17.1000
upper_band,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000,18.9000
